# ARXML Parser for Enums
This notebook parses ARXML files, extracts data type mappings, and generates a DataFrame with enumerated states.

In [2]:
# Import required libraries
import pandas as pd
from lxml import etree
import re
import os

In [3]:
# Define the parser function
def parse_arxml_with_enums(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        return pd.DataFrame()
        
    try:
        print(f"Parsing {file_path} (This might take a moment to build lookup tables)...")
        tree = etree.parse(file_path)
        root = tree.getroot()
    except etree.XMLSyntaxError as e:
        print(f"XML Parsing Error: {e}")
        return pd.DataFrame()

    # ==========================================
    # PRE-PROCESSING: Build Fast Lookup Dictionaries
    # ==========================================
    
    # 1. Extract all CompuMethods (The actual Enum dictionaries)
    compu_methods = {}
    for cm in root.xpath("//*[local-name()='COMPU-METHOD']"):
        cm_name_elem = cm.xpath("*[local-name()='SHORT-NAME']")
        if not cm_name_elem:
            continue
        cm_name = cm_name_elem[0].text.strip()
        
        enums = {}
        for scale in cm.xpath(".//*[local-name()='COMPU-SCALE']"):
            lower_limit = scale.xpath("*[local-name()='LOWER-LIMIT']")
            vt = scale.xpath(".//*[local-name()='VT']")
            if lower_limit and vt and lower_limit[0].text and vt[0].text:
                enums[lower_limit[0].text.strip()] = vt[0].text.strip()
        
        if enums:
            compu_methods[cm_name] = enums

    # 2. Extract Application Data Types to map them to CompuMethods
    app_to_compu = {}
    for app_dt in root.xpath("//*[local-name()='APPLICATION-PRIMITIVE-DATA-TYPE']"):
        app_name_elem = app_dt.xpath("*[local-name()='SHORT-NAME']")
        compu_ref = app_dt.xpath(".//*[local-name()='COMPU-METHOD-REF']")
        
        if app_name_elem and compu_ref and compu_ref[0].text:
            app_name = app_name_elem[0].text.strip()
            ref_name = compu_ref[0].text.split("/")[-1].strip()
            app_to_compu[app_name] = ref_name

    # Helper function to format the dictionary into a readable string
    def get_enum_string(app_type_name):
        compu_name = app_to_compu.get(app_type_name)
        enum_dict = compu_methods.get(compu_name, {})
        if not enum_dict:
            return "No Enums"
        return " | ".join([f"{k}: {v}" for k, v in enum_dict.items()])

    # ==========================================
    # MAIN PARSING: Generate the DataFrame
    # ==========================================
    parsed_data = []

    for dtms in root.xpath("//*[local-name()='DATA-TYPE-MAPPING-SET']"):
        short_name_elem = dtms.xpath("*[local-name()='SHORT-NAME']")
        if not short_name_elem:
            continue
        
        short_name = short_name_elem[0].text.strip()
        match = re.search(r'^X(\d+)_(.*?)SvcProv', short_name)
        if not match:
            continue
        
        sif = match.group(1)
        raw_event_name = match.group(2)
        someip_event = f"SomeIp{raw_event_name}"
        
        valid_methods = []
        valuestate_app_name = None
        
        for dt_map in dtms.xpath(".//*[local-name()='DATA-TYPE-MAP']"):
            app_ref = dt_map.xpath("*[local-name()='APPLICATION-DATA-TYPE-REF']")
            
            if app_ref and app_ref[0].text:
                app_path_raw = app_ref[0].text.split("/")[-1].strip()
                if re.match(r'^ValueState\d*$', app_path_raw, re.IGNORECASE):
                    valuestate_app_name = app_path_raw
                    continue 
                clean_method = app_path_raw[:-1] if app_path_raw.endswith("T") else app_path_raw
                valid_methods.append((clean_method, app_path_raw))
        
        for clean_method, raw_app_name in valid_methods:
            base_enums = get_enum_string(raw_app_name)
            parsed_data.append({
                "Cluster": "EthernetCluster",
                "SIF": sif,
                "Event": someip_event,
                "Method": clean_method,
                "Signal_String": f'"EthernetCluster::sif_{sif}::{someip_event}::{clean_method}"',
                "Available_States": base_enums
            })
            if valuestate_app_name:
                vs_event = someip_event[:-1] if someip_event.endswith('s') else someip_event
                vs_method = f"{clean_method}ValueState"
                vs_enums = get_enum_string(valuestate_app_name)
                parsed_data.append({
                    "Cluster": "EthernetCluster",
                    "SIF": sif,
                    "Event": vs_event,
                    "Method": vs_method,
                    "Signal_String": f'"EthernetCluster::sif_{sif}::{vs_event}::{vs_method}"',
                    "Available_States": vs_enums
                })

    df = pd.DataFrame(parsed_data)
    if not df.empty:
        df = df.sort_values(by=['SIF', 'Event', 'Method']).reset_index(drop=True)
    
    return df

In [4]:
# Example usage in notebook
file_name = "ETH_CAN.arxml"  # Replace with your ARXML file path
df_signals = parse_arxml_with_enums(file_name)

if not df_signals.empty:
    print(f"Success! Extracted {len(df_signals)} signals with state definitions.")
    
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', 50)
    display(df_signals.head(15))
    
    # Optionally export to CSV
    #df_signals.to_csv("signals_with_enums.csv", index=False)

Parsing ETH_CAN.arxml (This might take a moment to build lookup tables)...


Success! Extracted 7865 signals with state definitions.


,Cluster,SIF,Event,Method,Signal_String,Available_States
0,EthernetCluster,16414,SomeIpGadeSignal,GadeStatus,"""EthernetCluster::sif_16414::SomeIpGadeSignal:...",0: GADESTATUS_PARK_MODE | 1: GADESTATUS_LIFE_O...
1,EthernetCluster,16414,SomeIpGadeSignal,GadeStatusValueState,"""EthernetCluster::sif_16414::SomeIpGadeSignal:...",0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VA...
2,EthernetCluster,16414,SomeIpGadeSignal,gadeEvent,"""EthernetCluster::sif_16414::SomeIpGadeSignal:...",No Enums
3,EthernetCluster,16414,SomeIpGadeSignal,gadeEventValueState,"""EthernetCluster::sif_16414::SomeIpGadeSignal:...",0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VA...
4,EthernetCluster,16422,SomeIpMinVoltageReq,VoltageValueType,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
5,EthernetCluster,16422,SomeIpMinVoltageReq,VoltageValueTypeValueState,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VA...
6,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
7,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue1,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
8,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue1ValueState,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VA...
9,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue2,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums


In [6]:
# 1. Remove exact duplicate rows across all columns
# df = df_signals.drop_duplicates()

# (Optional: If you want to ensure uniqueness specifically based on the Signal String itself)
df = df_signals.drop_duplicates(subset=['Signal_String'])

# 2. Count how many signals have Enums vs. how many don't
# Note: Our script explicitly labeled missing enums as the string "No Enums"
missing_enums_count = (df['Available_States'] == "No Enums").sum()
has_enums_count = (df['Available_States'] != "No Enums").sum()

print(f"Total Unique Signals: {len(df)}")
print(f"Signals WITH Enums: {has_enums_count}")
print(f"Signals WITHOUT Enums (False Positives/Missing): {missing_enums_count}")

# 3. (Bonus) View a quick breakdown of exactly which enums are present
# This shows the top 10 most frequent enum states across your database
print("\n--- Top 10 Most Common Enum Mappings ---")
print(df['Available_States'].value_counts().head(10))

Total Unique Signals: 7795
Signals WITH Enums: 4558
Signals WITHOUT Enums (False Positives/Missing): 3237

--- Top 10 Most Common Enum Mappings ---
Available_States
0: VALUE_STATE_UNAVAILABLE | 1: VALUE_STATE_VALID | 2: VALUE_STATE_INVALID                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  3374
No Enums                                                                                                                                          

In [7]:
# Display a dataframe containing ONLY the signals missing their Enums
df_missing = df[df['Available_States'] == "No Enums"]
df_missing

,Cluster,SIF,Event,Method,Signal_String,Available_States
2,EthernetCluster,16414,SomeIpGadeSignal,gadeEvent,"""EthernetCluster::sif_16414::SomeIpGadeSignal:...",No Enums
4,EthernetCluster,16422,SomeIpMinVoltageReq,VoltageValueType,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
6,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
7,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue1,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
9,EthernetCluster,16422,SomeIpMinVoltageReq,minVoltageValue2,"""EthernetCluster::sif_16422::SomeIpMinVoltageR...",No Enums
...,...,...,...,...,...,...
7855,EthernetCluster,8387,SomeIpBodyRemoteConnectivity,Uuid6,"""EthernetCluster::sif_8387::SomeIpBodyRemoteCo...",No Enums
7857,EthernetCluster,8387,SomeIpBodyRemoteConnectivity,lsb6,"""EthernetCluster::sif_8387::SomeIpBodyRemoteCo...",No Enums
7859,EthernetCluster,8387,SomeIpBodyRemoteConnectivity,msb6,"""EthernetCluster::sif_8387::SomeIpBodyRemoteCo...",No Enums
7861,EthernetCluster,8387,SomeIpBodyRemoteConnectivity,setRHLBodyCmdRsp,"""EthernetCluster::sif_8387::SomeIpBodyRemoteCo...",No Enums
